# Recurrent Neural Network

## Part 1 - Data Preprocessing

### Importing the libraries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from datasets import load_dataset

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score

import tensorflow as tf
from keras.models import Sequential
from keras.layers import LSTM
from keras.layers import Dense
from keras.layers import Dropout
from keras.callbacks import EarlyStopping

### Importing the dataset

In [ ]:
dataset = load_dataset("GotThatData/kraken-trading-data")
data = dataset["train"].to_pandas()
ada_data = data[data["pair"] == "ADAUSD"].copy()
ada_data = ada_data.sort_values("timestamp")
features = ada_data[["price", "volume"]].values
print(features)

### Training/Test Split

In [ ]:
training_size = int(len(features) * 0.8)

train_set = features[:training_size]
test_set= features[training_size:]

print("\nTraining observations:", len(train_set))
print("Testing observations:", len(test_set))


### Feature Scaling

In [ ]:
sc = MinMaxScaler(feature_range = (0, 1))
train_set_scaled = sc.fit_transform(train_set)
test_set_scaled = sc.transform(test_set)

### Creating the time-series data structure with 60 timesteps 

In [ ]:
timesteps=60
X_train = []
y_train = []
for i in range(timesteps, len(train_set_scaled)):
    X_train.append(train_set_scaled[i-timesteps:i, :])
    y_train.append(train_set_scaled[i, 0])

X_train, y_train = np.array(X_train), np.array(y_train)

### Create test sequences

In [ ]:
test_input = np.concatenate(
    (train_set_scaled[-timesteps:], test_set_scaled), axis=0)

X_test = []
y_test = []

for i in range(timesteps, len(test_input)):

    X_test.append(test_input[i-timesteps:i, :])

    y_test.append(test_input[i, 0])


X_test, y_test = np.array(X_test), np.array(y_test)

### Display Data Shapes

In [ ]:
print("\nTraining shape:")
print(X_train.shape)

print("\ny_train shape:")
print(y_train.shape)

print("\nX_test shape:")
print(X_test.shape)

print("\ny_test shape:")
print(y_test.shape)

## Part 2 - Building and Training the RNN

### Initialising the RNN

In [ ]:
regressor = Sequential()

### Adding the first LSTM layer and some Dropout regularisation

In [ ]:
regressor.add(LSTM(units = 50, return_sequences = True, input_shape = (X_train.shape[1], X_train.shape[2])))
regressor.add(Dropout(0.2))

### Adding a second LSTM layer and some Dropout regularisation

In [ ]:
regressor.add(LSTM(units = 50, return_sequences = True))
regressor.add(Dropout(0.2))

### Adding a third LSTM layer and some Dropout regularisation

In [ ]:
regressor.add(LSTM(units = 50, return_sequences = True))
regressor.add(Dropout(0.2))

### Adding a fourth LSTM layer and some Dropout regularisation

In [ ]:
regressor.add(LSTM(units = 50))
regressor.add(Dropout(0.2))

### Adding the output layer

In [ ]:
regressor.add(Dense(units = 1))

### Compiling the RNN

In [ ]:
regressor.compile(optimizer = 'adam', loss = 'mean_squared_error')
regressor.summary()

### Early Stopping

In [ ]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)

### Fitting the RNN to the Training set

In [ ]:
regressor.fit(X_train, y_train, epochs = 100, batch_size = 32, validation_split=0.1, callbacks=[early_stop],
 shuffle=False)

## Part 3 - Making the predictions and visualising the results

### Converting predictions back to real price

In [ ]:
predicted_scaled = regressor.predict(X_test)

dummy_volume = np.zeros((len(predicted_scaled), 1))

predicted_with_volume = np.concatenate((predicted_scaled, dummy_volume), axis=1)

predicted_original = sc.inverse_transform(predicted_with_volume)

predicted_price = predicted_original[ :, 0].reshape(-1, 1)


### Convert actual prices back to original scale

In [ ]:
dummy_volume_actual = np.zeros((len(y_test), 1))

actual_scaled_with_volume = np.column_stack(( y_test,dummy_volume_actual[:, 0]))

actual_original = sc.inverse_transform( actual_scaled_with_volume)

actual_price = actual_original[:, 0].reshape(-1, 1)

### Visualising the results

In [ ]:

plt.figure(figsize=(14, 7))

plt.plot(
    actual_price,
    label="Real ADAUSD Price"
)

plt.plot(
    predicted_price,
    label="Predicted ADAUSD Price"
)

plt.title("ADAUSD Price Prediction")
plt.xlabel("Time")
plt.ylabel("ADAUSD Price")
plt.legend()

plt.show()

### Calculate Regression Metrics

In [ ]:
mae = mean_absolute_error(
    actual_price,
    predicted_price
)

rmse = np.sqrt(
    mean_squared_error(
        actual_price,
        predicted_price
    )
)

r2 = r2_score(
    actual_price,
    predicted_price
)

print("MAE:", mae)
print("RMSE:", rmse)
print("R²:", r2)